# LintGate Mutation Sweep

This notebook runs mutation profiling across the entire LintGate codebase using Colab's free compute.
It is **fully self-contained** — no external scripts needed.

**What it does:** For every Python function in `lintgate/` and `mcp_tools/`, it generates AST-level mutants
(value changes, argument swaps, boundary shifts, etc.) and runs your test suite against each mutant.
Functions where mutants *survive* (tests don't catch the change) have specification gaps.

**Results:** JSON files in `.lintgate/mutation/` — download and drop into your local project.
Then `mutation_get_state`, `mutation_prescribe`, and `spec_gate_check` all pick them up automatically.

---

## How to use this notebook

1. Click **Runtime > Run all** (or press `Ctrl+F9`). That's it.
2. Wait for all cells to finish (green checkmarks on the left).
3. The last cell downloads a zip file — unzip it into your local project root.

If the repo is **private**, you'll need to paste a GitHub token in Step 1 below.
If it's public, just run as-is.

## Step 1: Clone the repo and install

**If the repo is private:** Uncomment the `GITHUB_TOKEN` line below and paste your token.
You can create one at https://github.com/settings/tokens (only need `repo` scope).

**If public:** Just run as-is. The default branch is `main`.

In [ ]:
# === CONFIGURATION ===
REPO_URL = "https://github.com/rohanvinaik/LintGate.git"
BRANCH = "main"  # Change this if you need a different branch
PROJECT_DIR = "/content/lintgate"

# Uncomment and paste your token if the repo is private:
# GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"
# === END CONFIGURATION ===

import os, shutil, subprocess, sys

# Clean previous clone
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Build clone URL
clone_url = REPO_URL
try:
    GITHUB_TOKEN
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    print("Using authenticated clone (token set)")
except NameError:
    print("Using public clone (no token)")

# Clone
result = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, clone_url, PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"ERROR: git clone failed!\n{result.stderr}")
    print(f"\nCommon fixes:")
    print(f"  - Wrong branch? Try BRANCH = 'main'")
    print(f"  - Private repo? Set GITHUB_TOKEN above")
    raise RuntimeError("Clone failed — fix config above and re-run this cell")
print(f"Cloned {BRANCH} to {PROJECT_DIR}")

# Install
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "packaging"], check=True)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"pip install failed: {result.stderr}")
    # Fallback: just add to path
    print("Falling back to sys.path insertion")
    sys.path.insert(0, PROJECT_DIR)
else:
    # Force reimport after editable install
    for mod_name in list(sys.modules):
        if mod_name.startswith("lintgate"):
            del sys.modules[mod_name]

# Verify import works
try:
    from lintgate.specification.mutation_engine import MutationCategory
    print(f"Import OK. Mutation categories: {[c.value for c in MutationCategory]}")
except ImportError as e:
    # Last resort: direct path
    sys.path.insert(0, PROJECT_DIR)
    from lintgate.specification.mutation_engine import MutationCategory
    print(f"Import OK (via sys.path). Categories: {[c.value for c in MutationCategory]}")

print("\nReady for Step 2!")

## Step 2: Run the mutation sweep

This is the main event. Profiles every function in `lintgate/` and `mcp_tools/`.

**Expected time:** ~5-15 minutes for ~400 files on Colab free tier.

The sweep logic is embedded directly in this cell — no external scripts needed.

In [ ]:
import ast
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────
WORKERS = 4        # Parallel workers (4 is good for Colab free tier)
BUDGET_MS = 500.0  # Per-function mutation budget in milliseconds
SKIP_CACHED = True # Skip functions already profiled

# ── Collect source files ─────────────────────────────────────────
def collect_python_files(root):
    files = []
    for subdir in ("lintgate", "mcp_tools"):
        base = os.path.join(root, subdir)
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            for fn in sorted(filenames):
                if fn.endswith(".py") and not fn.startswith("test_") and not fn.endswith("_test.py"):
                    files.append(os.path.relpath(os.path.join(dirpath, fn), root))
    return files

# ── Profile a single file (runs in subprocess) ──────────────────
def profile_single_file(args):
    project_root, rel_path, budget_ms, skip_cached = args
    import ast, os, sys, time
    sys.path.insert(0, project_root)

    from lintgate.specification.mutation_engine import run_function_sampling
    from lintgate.keys import canonical_function_key
    from lintgate.specification.mutation_filter import filter_categories
    from mcp_tools._mutation_impl import (
        MutationContext, detect_purity_map, discover_test_files,
        get_cache_dir, load_test_callables, lookup_purity,
        parse_file, save_cached_state, walk_functions,
    )

    full_path = os.path.join(project_root, rel_path)
    cache_dir = get_cache_dir(project_root)
    start = time.monotonic()
    result = {"file": rel_path, "profiled": 0, "cached": 0, "trivial": 0, "errors": 0}

    tree = parse_file(full_path)
    if tree is None:
        result["errors"] = 1
        return result

    functions = walk_functions(tree)
    if not functions:
        return result

    test_files = discover_test_files(project_root, full_path)
    purity_map = detect_purity_map(full_path)
    ctx = MutationContext(
        full_path=full_path, rel_path=rel_path, cache_dir=cache_dir,
        purity_map=purity_map, test_files=test_files, project_root=project_root,
    )

    for qualname, node in functions:
        func_key = canonical_function_key(rel_path, qualname)

        if skip_cached:
            safe_key = func_key.replace("::", "__").replace("/", "_")
            if (cache_dir / f"{safe_key}.json").exists():
                result["cached"] += 1
                continue

        body = getattr(node, "body", [])
        if len(body) <= 1:
            stmt = body[0] if body else None
            if isinstance(stmt, (ast.Return, ast.Expr)):
                result["trivial"] += 1
                continue

        is_pure = lookup_purity(purity_map, qualname)
        cats = filter_categories(node, is_pure=is_pure)
        bare_name = qualname.split(".")[-1]
        tests, _ = load_test_callables(
            ctx.test_files, bare_name,
            project_root=ctx.project_root, func_key=func_key,
        )

        try:
            sr = run_function_sampling(
                node, func_key, cats, tests, lambda *_: None, budget_ms=budget_ms,
            )
            rd = sr.to_dict()
            rd["tests_loaded"] = len(tests)
            rd["is_pure"] = is_pure
            args_node = getattr(node, "args", None)
            rd["parameter_count"] = len(args_node.args) if args_node else 0
            save_cached_state(ctx.cache_dir, func_key, rd)
            result["profiled"] += 1
        except Exception:
            result["errors"] += 1

    result["elapsed_s"] = round(time.monotonic() - start, 2)
    return result

# ── Run the sweep ────────────────────────────────────────────────
project_root = os.path.abspath(PROJECT_DIR)
files = collect_python_files(project_root)
print(f"Found {len(files)} source files | Workers: {WORKERS} | Budget: {BUDGET_MS}ms")

cache_dir = Path(project_root) / ".lintgate" / "mutation"
cache_dir.mkdir(parents=True, exist_ok=True)

work = [(project_root, f, BUDGET_MS, SKIP_CACHED) for f in files]
totals = {"profiled": 0, "cached": 0, "trivial": 0, "errors": 0}
start = time.monotonic()

with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(profile_single_file, w): w[1] for w in work}
    for i, future in enumerate(as_completed(futures), 1):
        rel = futures[future]
        try:
            r = future.result()
            for k in totals:
                totals[k] += r.get(k, 0)
            print(f"[{i}/{len(files)}] {rel}: {r.get('profiled',0)} profiled, "
                  f"{r.get('cached',0)} cached, {r.get('elapsed_s','?')}s")
        except Exception as e:
            print(f"[{i}/{len(files)}] {rel}: FAILED - {e}")
            totals["errors"] += 1

elapsed = round(time.monotonic() - start, 1)
print(f"\n{'='*60}")
print(f"Done in {elapsed}s")
print(f"  Profiled: {totals['profiled']}")
print(f"  Cached:   {totals['cached']}")
print(f"  Trivial:  {totals['trivial']}")
print(f"  Errors:   {totals['errors']}")

## Step 3: Review results

In [ ]:
import json, os
from pathlib import Path

cache_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
if not cache_dir.exists():
    print("No results found. Did Step 4 complete?")
else:
    files = sorted(cache_dir.glob('*.json'))
    print(f"Total cached profiles: {len(files)}")
    print()
    
    # Aggregate stats
    total_funcs = 0
    total_survived = 0
    total_killed = 0
    high_survival = []  # functions with >50% survival
    
    for f in files:
        try:
            data = json.loads(f.read_text())
        except (json.JSONDecodeError, OSError):
            continue
        
        total_funcs += 1
        survived = data.get('total_survived', 0)
        killed = data.get('total_killed', 0)
        total_survived += survived
        total_killed += killed
        
        rate = data.get('survival_rate', 0)
        if rate > 0.5 and (survived + killed) > 0:
            high_survival.append((data.get('function_key', '?'), rate, survived + killed))
    
    total_mutants = total_killed + total_survived
    overall_kill_rate = total_killed / total_mutants if total_mutants else 0
    
    print(f"Functions profiled: {total_funcs}")
    print(f"Total mutants:     {total_mutants}")
    print(f"Killed:            {total_killed} ({overall_kill_rate:.1%})")
    print(f"Survived:          {total_survived} ({1-overall_kill_rate:.1%})")
    print()
    
    if high_survival:
        high_survival.sort(key=lambda x: -x[1])
        print(f"Functions with >50% survival rate ({len(high_survival)} total):")
        print(f"{'Function':<70} {'Rate':>6} {'Mutants':>8}")
        print('-' * 86)
        for key, rate, count in high_survival[:25]:
            # Truncate long keys
            display_key = key if len(key) <= 68 else '...' + key[-65:]
            print(f"{display_key:<70} {rate:>5.1%} {count:>8}")
        if len(high_survival) > 25:
            print(f"  ... and {len(high_survival) - 25} more")
    else:
        print("No functions with >50% survival. Your test suite is solid!")

## Step 4: Download results

Downloads a zip file. Then on your local machine:

```bash
cd /Users/rohanvinaik/tools/lintgate
unzip ~/Downloads/mutation_results.zip
```

After that, all MCP tools (`mutation_get_state`, `mutation_prescribe`, `spec_gate_check`, etc.)
will see the Colab-computed results immediately.

In [ ]:
import os
from pathlib import Path

# Ensure cache dir exists before zipping
mutation_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
mutation_dir.mkdir(parents=True, exist_ok=True)

# Create the zip
zip_path = '/content/mutation_results.zip'
!cd {PROJECT_DIR} && zip -r {zip_path} .lintgate/mutation/ -x '*.DS_Store'

# Show zip size and download
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"\nZip size: {size_mb:.1f} MB")
    try:
        from google.colab import files
        files.download(zip_path)
        print("Download started! Check your browser's download bar.")
    except ImportError:
        print(f"Not in Colab. Results at: {zip_path}")
else:
    print("No zip created. Check that the sweep in Step 4 produced results.")

print(f"\nTo apply locally:")
print(f"  cd /Users/rohanvinaik/tools/lintgate")
print(f"  unzip ~/Downloads/mutation_results.zip")

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `git clone` fails with 403 | Repo is private. Set `GITHUB_TOKEN` in Step 1 |
| `ModuleNotFoundError` | Step 2 didn't complete. Re-run it |
| Sweep seems stuck | Check Colab RAM (top-right). If maxed out, reduce `MUTATION_WORKERS` to 2 |
| "Session crashed" | Colab ran out of memory. Reduce workers or budget |
| Many errors in sweep | Normal for functions with complex imports. The script continues past errors |
| Want to re-run on new code | Set `MUTATION_SKIP_CACHED = '0'` in Step 4, or delete `.lintgate/mutation/` and re-run |